In [9]:
# Step 1. 필요한 모듈과 라이브러리를 로딩합니다.
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time   # 시간 관련 모듈
import sys    # txt파일 저장관련 모듈
import math   # 페이지 번호 계산위해 사용 모듈
import pandas as pd   
import os
import urllib.request
import urllib

# Step 2. 사용자에게 검색어 키워드를 입력 받습니다.
print("=" *80)
print("지마켓 Best Seller 상품 정보 추출하기 ")
print("=" *80)

query_txt = 'G마켓'
query_url='http://corners.gmarket.co.kr/Bestsellers'

cnt = int(input('1.크롤링 할 건수는 몇건입니까?: '))
page_cnt = math.ceil(cnt / 60)

f_dir = input('2.파일을 저장할 폴더명만 쓰세요(기본값:c:\\py_temp\\):')
if f_dir =='' :
    f_dir = "c:\\py_temp\\"

# Step 3.저장될 파일위치와 이름을 지정합니다
now = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' %(now.tm_year, now.tm_mon, now.tm_mday, \
                                      now.tm_hour, now.tm_min, now.tm_sec)

img_dir = f_dir+s+'-'+query_txt+"\\images"

os.makedirs(img_dir)
os.chdir(f_dir+s+'-'+query_txt)

ff_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.txt'
fc_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.csv'
fx_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.xlsx'

#Step 4. 크롬 드라이버를 지정하고 웹사이트에 접속합니다
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

driver.get(query_url)
time.sleep(1)

def scroll_down(driver):      
    driver.execute_script("window.scrollBy(0,2500);")
    time.sleep(4)
    
for i in range(1,4) :
    scroll_down(driver)
        
#Step 5.수집 내용을 저장하기 위해 빈 리스트를 지정합니다
ranking2=[]    # 판매 순위 지정용 리스트
title2=[]      # 제품 이름 저장용 리스트
o_price2=[]    # 원래 가격 저장용 리스트
s_price2=[]    # 현재판매가격 저장용 리스트
discount2=[]   # 할인율 저장용 리스트

#Step 6. 웹 페이지에서 정보를 추출합니다
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')
slist = soup.find('ul','list__best').find_all('li')

#Step 7-1.이미지 추출하기
img_file_no = 1

for li in slist:
                
    os.chdir(img_dir)

    try :
          photo = li.find('div','box__thumbnail').find('img')['src']
    except AttributeError :
        continue
    
    print('%s번째 이미지를 저장하고 있으니 잠시 기다려 주세요~~' %img_file_no)
    photo2 = 'http:' + photo
    urllib.request.urlretrieve(photo2,str(img_file_no)+'.jpg')
    time.sleep(2)

    img_file_no += 1

    if img_file_no > cnt :
        break 

#Step 7-2. 텍스트 추출하기
count = 1   # 판매순위용 변수
for li in slist:
            
    f = open(ff_name, 'a',encoding='UTF-8')
    f.write("-----------------------------------------------------"+"\n")
    print("-" *70)
    
    #판매순위
    sid = '#no' + str(count)
    try :
        ranking = li.find('div','box__thumbnail').get_text()
    except AttributeError :
        ranking = ''
    
    print('1.판매순위:',ranking.replace("\n",""))
    f.write('1.판매순위:'+ ranking + "\n")
    ranking2.append(ranking)

    #제품이름
    try :
        title = li.find('div','box__item-info').find('p','box__item-title').get_text()
    except AttributeError :
        title = '제품명이없습니다'

    print("2.제품이름:", title.replace("\n",""))
    f.write('2.제품이름:'+ title + "\n")
    title2.append(title.replace("\n",""))

    #원래 가격
    try:
        o_price = li.find('div', 'box__price-original').find('span','text text__value').get_text()
    except :
        o_price = li.find('div','box__price-seller').find('span','text text__value').get_text()
        
    print("3.원래가격:", o_price.replace("\n",""))
    f.write('3.원래가격:'+ o_price + "\n")
    o_price2.append(o_price.replace("\n",""))
    
    #판매 가격
    try :
        s_price = li.find('div', 'box__price-seller').find('span','text text__value').get_text()
    except :
        s_price = li.find('div','box__price-seller').find('span','text text__value').get_text()
        
    print("4.판매가격:", s_price.replace("\n",""))
    f.write('4.판매가격:'+ s_price + "\n")
    s_price2.append(s_price.replace("\n",""))

    #할인율
    try :
        discount = li.find('div', 'box__discount').find('span','text text__value').get_text()
    except :
        discount = '0%'

    print("5.할인율:", discount.replace("\n",""))
    f.write('5.할인율:'+ discount + "\n")
    discount2.append(discount.replace("\n",""))     

    if count == cnt :
        break

    count += 1

    time.sleep(0.5)

#step 8. csv, xlsx 형태의 파일로  저장하기              
g_best_seller = pd.DataFrame()

g_best_seller['판매순위']=ranking2
g_best_seller['제품소개']=pd.Series(title2)
g_best_seller['원래가격']=pd.Series(o_price2)
g_best_seller['판매가격']=pd.Series(s_price2)
g_best_seller['할인율']=pd.Series(discount2)

# csv 형태로 저장하기
g_best_seller.to_csv(fc_name,encoding="utf-8-sig",index=False)

# 엑셀 형태로 저장하기
g_best_seller.to_excel(fx_name ,index=False)

# 요약정보 출력하기  
print("\n") 
print("=" *80)
print("1.요청된 총 %s 건의 리뷰 중에서 실제 크롤링 된 건수는 %s 건입니다" %(cnt,count))
print("2.파일 저장 완료: txt 파일명 : %s " %ff_name)
print("3.파일 저장 완료: csv 파일명 : %s " %fc_name)
print("4.파일 저장 완료: xlsx 파일명 : %s " %fx_name)
print("=" *80)
     
#Step 9.이미지 삽입하기
import win32com.client as win32   
import win32api                
excel = win32.gencache.EnsureDispatch('Excel.Application')
wb = excel.Workbooks.Open(fx_name)
sheet = wb.ActiveSheet
sheet.Columns(3).ColumnWidth = 30   #  이미지 가로 사이즈에 맞게 컬럼 크기 조정
row_cnt = cnt+1
sheet.Rows("2:%s" %row_cnt).RowHeight = 120  #  이미지 세로 사이즈에 맞게 로우 크기 조정

ws = wb.Sheets("Sheet1")
col_name2=[]
file_name2=[]

for a in range(2,cnt+2) :
    col_name='C'+str(a)
    col_name2.append(col_name)

for b in range(1,cnt+1) :
    file_name=img_dir+'\\'+str(b)+'.jpg'
    file_name2.append(file_name)
      
for i in range(0,cnt) :
    rng = ws.Range(col_name2[i])
    image = ws.Shapes.AddPicture(file_name2[i], False, True, rng.Left, rng.Top, 130, 100)
    excel.Visible=True
    excel.ActiveWorkbook.Save()

driver.close()

지마켓 Best Seller 상품 정보 추출하기 


1.크롤링 할 건수는 몇건입니까?:  10
2.파일을 저장할 폴더명만 쓰세요(기본값:c:\py_temp\): 


1번째 이미지를 저장하고 있으니 잠시 기다려 주세요~~
2번째 이미지를 저장하고 있으니 잠시 기다려 주세요~~
3번째 이미지를 저장하고 있으니 잠시 기다려 주세요~~
4번째 이미지를 저장하고 있으니 잠시 기다려 주세요~~
5번째 이미지를 저장하고 있으니 잠시 기다려 주세요~~
6번째 이미지를 저장하고 있으니 잠시 기다려 주세요~~
7번째 이미지를 저장하고 있으니 잠시 기다려 주세요~~
8번째 이미지를 저장하고 있으니 잠시 기다려 주세요~~
9번째 이미지를 저장하고 있으니 잠시 기다려 주세요~~
10번째 이미지를 저장하고 있으니 잠시 기다려 주세요~~

----------------------------------------------------------------------
1.판매순위: 1
2.제품이름: 다향오리 훈제슬라이스 200gx10팩
3.원래가격: 42,500
4.판매가격: 31,880
5.할인율: 24%

----------------------------------------------------------------------
1.판매순위: 2
2.제품이름: 하남쭈꾸미쭈꾸미볶음 500g 3팩
3.원래가격: 35,900
4.판매가격: 28,720
5.할인율: 20%

----------------------------------------------------------------------
1.판매순위: 3
2.제품이름: 스마일 뜯어쓰는 스케치북 8절 도화지 250매 130g
3.원래가격: 25,000
4.판매가격: 8,990
5.할인율: 64%

----------------------------------------------------------------------
1.판매순위: 4
2.제품이름: (신선집중) 호주산 와규 바로구이 대패 1kg
3.원래가격: 27,400
4.판매가격: 17,520
5.할인율: 36%

--------------------------------------------------------------------